# Week 02 — Python Solution Lab
## 1D Kinematics

**Companion to `notebooks/Week_02.ipynb`.** This notebook contains *fully worked Python
solutions* to selected problems from that week's problem set — one at **each difficulty level**.

Every solution follows the course's core workflow:

> **Diagram → Principle → Equation → Predict → Verify**

The markdown cell states the problem, identifies the governing principle, and gives the **hand
prediction you should make before running anything**. The code cell then computes the result and
*verifies* it — typically by a second independent method (energy vs. forces, symbolic vs.
numerical, closed form vs. simulation) — and includes `assert` checks against the known answer.

### How to use this notebook

1. **Attempt the problem in `Week_02.ipynb` first.** These solutions are worth very little
   if you read them before trying.
2. Make the hand prediction. Write it down.
3. Run the code cell and compare.
4. **Change a number and re-run.** Every solution is written so that the parameters sit at the
   top; the sweeps and plots update automatically. Ask "what if the mass doubled?" and answer it
   in ten seconds.

### Why the code looks like this

These are not minimal answer-generators. Each one demonstrates something Python does that hand
algebra cannot: parameter sweeps, root-finding, numerical integration, symbolic differentiation,
or a cross-check to machine precision. The physics is the point; the code is how we prove the
physics is right.

---


### Solutions in this notebook

| Level | Problem | Topic | Python technique |
|---|---|---|---|
| **L1 · Basic** | `P3` | Free Fall Drop | 3 routes: kinematics / energy / simulation |
| **L2 · Intermediate** | `P7` | Braking Distance Analysis | parameter sweep to a critical value |
| **L3 · Challenge** | `P9` | Position Function Analysis | **SymPy** symbolic differentiation |

---

## L1 · Basic — P3: Free Fall Drop

> **Problem (Week_02.ipynb, L1 — P3).** A stone is dropped from rest from the top of a
> $45.0$ m tall building. Ignoring air resistance, (a) how long does it take to reach the
> ground? (b) What is its speed just before impact?

**Diagram → Principle.** Constant acceleration $a = -g$, initial velocity zero.

**Equation.** $h = \tfrac12 g t^2 \Rightarrow t = \sqrt{2h/g}$, then $v = gt$.

**Hand prediction.** $t = \sqrt{90/9.81} = 3.03$ s, $v = 29.7$ m/s.

**What Python adds.** We solve it in closed form, then **re-derive the same numbers from a
time-stepped simulation** and from energy conservation. Three independent routes agreeing to
four digits is what "verify" means in this course — and the $y(t)$, $v(t)$ plot makes the
parabola-versus-line relationship concrete.

In [ ]:
# ═══ W02 · L1 · P3 — Free fall: closed form, simulation, and energy ═══
import numpy as np
import matplotlib.pyplot as plt

# --- MODEL --------------------------------------------------------------
h, g, v0 = 45.0, 9.81, 0.0      # m, m/s^2, m/s

# --- PREDICT (route 1: kinematic equations) -----------------------------
t_fall   = np.sqrt(2 * h / g)
v_impact = g * t_fall
print(f"(a) t = sqrt(2h/g) = {t_fall:.3f} s")
print(f"(b) v = g*t        = {v_impact:.2f} m/s  ({v_impact * 3.6:.0f} km/h)")

# --- VERIFY (route 2: energy conservation, no time involved) ------------
v_energy = np.sqrt(2 * g * h)
print(f"\nEnergy route:  v = sqrt(2gh) = {v_energy:.2f} m/s   (agrees, and never used t)")
assert np.isclose(v_impact, v_energy)

# --- VERIFY (route 3: numerical integration on a fine time grid) --------
t  = np.linspace(0, t_fall, 20001)
y  = h - 0.5 * g * t**2
v  = -g * t
i_hit = np.argmin(np.abs(y))                    # first grid point at the ground
print(f"Simulation:    ground reached at t = {t[i_hit]:.3f} s, v = {abs(v[i_hit]):.2f} m/s")
assert abs(t[i_hit] - t_fall) < 1e-3

# --- Plot ---------------------------------------------------------------
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9.5, 3.6))
ax1.plot(t, y, color="#1565c0", lw=2); ax1.set_ylabel("y (m)")
ax1.set_title("position: parabola")
ax2.plot(t, v, color="#e65100", lw=2); ax2.set_ylabel("v (m/s)")
ax2.set_title("velocity: straight line, slope = -g")
for ax in (ax1, ax2):
    ax.set_xlabel("t (s)"); ax.grid(alpha=.3); ax.axhline(0, c="k", lw=.6)
plt.suptitle("W02 P3 — 45 m free fall", y=1.02); plt.tight_layout(); plt.show()

# --- Bonus: what would air resistance do? (looking ahead to Week 14) ----
print(f"\nNote: with air resistance the fall would take longer than {t_fall:.2f} s and the")
print("impact speed would be lower. How much depends on the stone's mass, size, shape and")
print("drag coefficient -- there is no single 'terminal speed for a stone'. We drop the")
print("drag term now and put it back in the Week 14 capstone.")

# --- CHECK --------------------------------------------------------------
assert abs(t_fall - 3.03) < 0.01 and abs(v_impact - 29.7) < 0.1
print("[OK] Matches textbook answer: t = 3.03 s, v = 29.7 m/s")

## L2 · Intermediate — P7: Braking Distance Analysis

> **Problem (Week_02.ipynb, L2 — P7).** A car travels at $108$ km/h when the driver sees an
> obstacle $120$ m ahead. Reaction time is $0.80$ s and braking deceleration is $9.0$ m/s².
> (a) Reaction distance? (b) Total stopping distance? (c) Does the car stop in time?

**Diagram → Principle.** Two phases: constant velocity during reaction, then constant
deceleration. Total distance is the sum.

**Equation.** $d = v_0 t_r + \dfrac{v_0^2}{2a}$.

**Hand prediction.** $d_{\text{react}} = 24$ m, $d_{\text{brake}} = 50$ m, total $74$ m $< 120$ m — it stops.

**What Python adds.** The single yes/no answer hides the real engineering question: *how much
margin do you have, and where does it disappear?* We sweep initial speed to find the exact
**critical speed** at which 120 m is no longer enough, and show the quadratic term is what
kills you — doubling speed roughly quadruples the braking distance.

In [ ]:
# ═══ W02 · L2 · P7 — Braking distance, and the speed at which the margin vanishes ═══
import numpy as np
import matplotlib.pyplot as plt

# --- MODEL --------------------------------------------------------------
v0     = 108 / 3.6      # m/s  (108 km/h)
t_r    = 0.80           # s, reaction time
a      = 9.0            # m/s^2, braking deceleration (magnitude)
d_obst = 120.0          # m

def stopping_distance(v, t_react=t_r, decel=a):
    """Reaction leg (linear in v) + braking leg (quadratic in v)."""
    return v * t_react + v**2 / (2 * decel)

# --- PREDICT ------------------------------------------------------------
d_react = v0 * t_r
d_brake = v0**2 / (2 * a)
d_total = d_react + d_brake
print(f"v0 = {v0:.1f} m/s")
print(f"(a) reaction distance = {d_react:.1f} m")
print(f"    braking  distance = {d_brake:.1f} m")
print(f"(b) total stopping    = {d_total:.1f} m")
print(f"(c) obstacle at {d_obst:.0f} m -> "
      f"{'STOPS IN TIME' if d_total < d_obst else 'COLLISION'}, margin = {d_obst - d_total:.1f} m")

# --- VERIFY: sweep speed to find the critical value ---------------------
speeds = np.linspace(0, 45, 2000)                 # m/s
dists  = stopping_distance(speeds)
v_crit = speeds[np.argmax(dists > d_obst)]        # first speed that overruns
print(f"\nCritical speed for a 120 m gap: {v_crit:.2f} m/s = {v_crit * 3.6:.1f} km/h")
print(f"  -> only {v_crit * 3.6 - 108:.0f} km/h faster than the stated speed. Thin margin.")

# --- VERIFY: the quadratic term dominates -------------------------------
print("\n  v (km/h) | react (m) | brake (m) | total (m)")
for kmh in (50, 100, 150, 200):
    v = kmh / 3.6
    print(f"   {kmh:6.0f} | {v*t_r:9.1f} | {v**2/(2*a):9.1f} | {stopping_distance(v):8.1f}")
print("  Reaction distance doubles with speed; braking distance QUADRUPLES.")

# --- Plot ---------------------------------------------------------------
fig, ax = plt.subplots(figsize=(7.4, 4))
ax.plot(speeds * 3.6, speeds * t_r, label="reaction (linear)", color="#1565c0", ls="--")
ax.plot(speeds * 3.6, speeds**2 / (2*a), label="braking (quadratic)", color="#2e7d32", ls="--")
ax.plot(speeds * 3.6, dists, label="total", color="#e65100", lw=2.5)
ax.axhline(d_obst, c="crimson", ls=":", lw=2, label=f"obstacle at {d_obst:.0f} m")
ax.axvline(108, c="grey", ls=":", label="108 km/h (this problem)")
ax.plot(v_crit * 3.6, d_obst, "o", color="crimson", ms=9, zorder=5)
ax.set_xlabel("initial speed (km/h)"); ax.set_ylabel("distance (m)")
ax.set_title("W02 P7 — why speed limits are quadratic, not linear")
ax.set_ylim(0, 260); ax.grid(alpha=.3); ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

# --- CHECK --------------------------------------------------------------
assert abs(d_react - 24.0) < 0.1 and abs(d_brake - 50.0) < 0.1
assert abs(d_total - 74.0) < 0.1 and d_total < d_obst
print("[OK] Matches textbook answer: 24 m + 50 m = 74 m < 120 m, the car stops in time.")

## L3 · Challenge — P9: Position Function Analysis

> **Problem (Week_02.ipynb, L3 — P9).** The position of a particle is
> $x(t) = 4.0t^3 - 6.0t^2 + 2.0t$ (SI units). (a) Derive $v(t)$ and $a(t)$. (b) When is the
> velocity zero? (c) Position and acceleration at those times? (d) Describe the motion.

**Diagram → Principle.** $v = dx/dt$, $a = dv/dt$. The "turning points" are the roots of $v(t)$.

**Equation.** $v(t) = 12t^2 - 12t + 2$, $a(t) = 24t - 12$.

**Hand prediction.** $v=0$ at $t = \frac{12 \pm \sqrt{144-96}}{24} = 0.211$ s and $0.789$ s.

**What Python adds.** This is the natural place to introduce **SymPy**. We differentiate
symbolically (no algebra slips), solve $v(t)=0$ exactly, and only *then* substitute numbers.
The same script would handle a degree-7 polynomial unchanged. We cross-check the symbolic
derivative against a numerical one, and read the qualitative description (d) straight off the
plot rather than guessing it.

In [ ]:
# ═══ W02 · L3 · P9 — Symbolic calculus with SymPy, checked numerically ═══
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt

# --- MODEL: define x(t) symbolically ------------------------------------
t = sp.symbols('t', real=True)
x = 4*t**3 - 6*t**2 + 2*t

# --- (a) PREDICT: differentiate, don't hand-expand ----------------------
v = sp.diff(x, t)
a = sp.diff(v, t)
print(f"(a) x(t) = {x}")
print(f"    v(t) = {v}")
print(f"    a(t) = {a}")

# --- (b) exact roots of v(t) = 0 ----------------------------------------
roots = sorted(sp.solve(sp.Eq(v, 0), t))
print(f"\n(b) v = 0 exactly at t = {roots}")
print(f"    numerically: {[float(r) for r in roots]}")

# --- (c) evaluate x and a at those instants -----------------------------
print("\n(c)      t (s)      x (m)      a (m/s^2)")
for r in roots:
    print(f"    {float(r):9.4f}  {float(x.subs(t, r)):9.4f}  {float(a.subs(t, r)):11.4f}")

# --- VERIFY: symbolic derivative vs a numerical one ---------------------
x_f = sp.lambdify(t, x, "numpy")
v_f = sp.lambdify(t, v, "numpy")
a_f = sp.lambdify(t, a, "numpy")
tg  = np.linspace(-0.2, 1.2, 1401)
v_numeric = np.gradient(x_f(tg), tg)
err = np.max(np.abs(v_numeric[5:-5] - v_f(tg)[5:-5]))
print(f"\nmax |numeric dv - symbolic v| = {err:.2e} m/s  -> the derivative is right")
assert err < 1e-3

# --- (d) read the qualitative description off the graph -----------------
t1, t2 = float(roots[0]), float(roots[1])
print(f"\n(d) t < {t1:.3f}: v > 0, moving +x and slowing (a < 0)")
print(f"    {t1:.3f} < t < {t2:.3f}: v < 0, moving BACKWARD  (x falls from "
      f"{float(x.subs(t, roots[0])):.4f} m to {float(x.subs(t, roots[1])):.4f} m)")
print(f"    t > {t2:.3f}: v > 0 again and a > 0, accelerating away in +x")
print(f"    Sign of a flips at t = {float(sp.solve(sp.Eq(a,0), t)[0]):.3f} s (the inflection).")

fig, axes = plt.subplots(3, 1, figsize=(7, 7), sharex=True)
for ax, f, lbl, col in zip(axes, (x_f, v_f, a_f),
                           ("x (m)", "v (m/s)", "a (m/s$^2$)"),
                           ("#1565c0", "#e65100", "#2e7d32")):
    ax.plot(tg, f(tg) * np.ones_like(tg), color=col, lw=2)
    ax.set_ylabel(lbl); ax.grid(alpha=.3); ax.axhline(0, c="k", lw=.6)
    for r in (t1, t2):
        ax.axvline(r, c="grey", ls=":")
axes[0].set_title("W02 P9 — x, v, a for $x(t)=4t^3-6t^2+2t$ (dotted: v = 0)")
axes[-1].set_xlabel("t (s)")
plt.tight_layout(); plt.show()

# --- CHECK --------------------------------------------------------------
assert abs(t1 - 0.2113) < 1e-3 and abs(t2 - 0.7887) < 1e-3
assert sp.simplify(v - (12*t**2 - 12*t + 2)) == 0
print("[OK] v(t) = 12t^2 - 12t + 2, a(t) = 24t - 12, v = 0 at t = 0.211 s and 0.789 s")

---

## Self-check

**Every code cell above contains one or more `assert` checks.** If you run the whole notebook top
to bottom without an `AssertionError`, all of the numerical checks on this page have passed.
(The asserts sit just before each cell's closing summary, so the last thing you see is a printed
result — not the check itself.)

**Now transfer the skill.** Pick one unsolved problem from `Week_02.ipynb` at the level you
found hardest, and write the same five-part structure for it:

```python
# --- MODEL:   parameters at the top, with units in comments
# --- PREDICT: the closed-form answer
# --- VERIFY:  a SECOND, independent route to the same number
# --- CHECK:   assert against your hand prediction
```

The verify step is the one that matters. A result you have only computed one way is a result you
have not checked.
